# The Enigma Machine and the Bombe

The Enigma machine was an electro-mechanical rotor cipher used during World War II. A letter entered through a plugboard, moved through spinning rotors, bounced off a reflector, and came back through the rotors to produce a scrambled letter.

Breaking Enigma joined mathematics, engineering, linguistics, and disciplined operational analysis. The Polish Cipher Bureau and later Bletchley Park showed how a cipher machine can be attacked by exploiting structure, repeated patterns, and mechanical constraints.

This notebook has two parts:

- Build a working Enigma-style rotor machine.
- Build a simplified Bombe search that uses a guessed word, called a crib, to find the rotor setting.

<details>
<summary>Big idea</summary>

Enigma is not just substitution. The substitution changes after every keypress because the rotors move. That makes the same plaintext letter encrypt differently in different positions.

</details>

## 1. Mental Model

A simplified Enigma machine has these pieces:

- **Plugboard**: swaps letter pairs before and after the rotor path.
- **Rotors**: shifting substitution wheels. The right rotor steps every keypress.
- **Reflector**: sends the signal back through the rotors.
- **Starting positions**: the secret key for this lesson.

<details>
<summary>Why encryption and decryption are the same</summary>

The reflector makes the circuit reversible. If you reset the machine to the same starting settings and type the ciphertext, the plaintext comes back out.

</details>

## 2. Build the Machine Objects

We will model Enigma with small Python objects:

- `RotorSpec` stores the historical wiring for a rotor.
- `Rotor` tracks the rotor position and maps letters forward/backward.
- `Plugboard` swaps letter pairs.
- `Reflector` bounces the signal back.
- `EnigmaMachine` steps the rotors and records a path snapshot for each letter.

<details>
<summary>Implementation hint</summary>

A rotor is easiest to code as an alphabet index shift. Convert `A` to `0`, `B` to `1`, and so on, then use modulo arithmetic to wrap around the alphabet.

</details>

**Object model.** Define `RotorSpec`, the named objects used by the next examples.


In [ ]:
from dataclasses import dataclass, field

from itertools import product

ALPHABET = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"

LETTER_TO_INDEX = {letter: index for index, letter in enumerate(ALPHABET)}

def normalize_message(message: str) -> str:
    return "".join(character for character in message.upper() if character in ALPHABET)

def letter_at(index: int) -> str:
    return ALPHABET[index % 26]

@dataclass(frozen=True)
class RotorSpec:
    name: str
    wiring: str
    notch: str

    def __post_init__(self) -> None:
        if sorted(self.wiring) != sorted(ALPHABET):
            raise ValueError(f"Rotor {self.name} wiring must contain each letter once.")
        if self.notch not in ALPHABET:
            raise ValueError("Rotor notch must be a single capital letter.")


**Object model.** Define `Rotor`, the named objects used by the next examples.


In [ ]:
@dataclass
class Rotor:
    spec: RotorSpec
    position: int = 0
    ring_setting: int = 0

    @classmethod
    def from_letter(cls, spec: RotorSpec, position: str = "A", ring_setting: str = "A") -> "Rotor":
        return cls(
            spec=spec,
            position=LETTER_TO_INDEX[position],
            ring_setting=LETTER_TO_INDEX[ring_setting],
        )

    @property
    def position_letter(self) -> str:
        return letter_at(self.position)

    def at_notch(self) -> bool:
        return self.position_letter == self.spec.notch

    def step(self) -> None:
        self.position = (self.position + 1) % 26

    def encode_forward(self, letter: str) -> str:
        shifted_index = (LETTER_TO_INDEX[letter] + self.position - self.ring_setting) % 26
        wired_letter = self.spec.wiring[shifted_index]
        return letter_at(LETTER_TO_INDEX[wired_letter] - self.position + self.ring_setting)

    def encode_backward(self, letter: str) -> str:
        shifted_index = (LETTER_TO_INDEX[letter] + self.position - self.ring_setting) % 26
        contact_letter = letter_at(shifted_index)
        wired_index = self.spec.wiring.index(contact_letter)
        return letter_at(wired_index - self.position + self.ring_setting)


**Object model.** Define `Plugboard`, the named objects used by the next examples.


In [ ]:
@dataclass
class Plugboard:
    pairs: tuple[str, ...] = ()
    mapping: dict[str, str] = field(init=False)

    def __post_init__(self) -> None:
        mapping = {letter: letter for letter in ALPHABET}
        used_letters: set[str] = set()

        for pair in self.pairs:
            clean_pair = pair.upper()
            if len(clean_pair) != 2 or any(letter not in ALPHABET for letter in clean_pair):
                raise ValueError(f"Invalid plugboard pair: {pair}")
            left, right = clean_pair
            if left == right or left in used_letters or right in used_letters:
                raise ValueError(f"Plugboard letter reused in pair: {pair}")
            mapping[left] = right
            mapping[right] = left
            used_letters.update((left, right))

        self.mapping = mapping

    def swap(self, letter: str) -> str:
        return self.mapping[letter]


**Trace model.** Define `Reflector`, `EnigmaSnapshot`, the structure used to capture replayable algorithm state.


In [ ]:
@dataclass(frozen=True)
class Reflector:
    wiring: str

    def reflect(self, letter: str) -> str:
        return self.wiring[LETTER_TO_INDEX[letter]]

@dataclass(frozen=True)
class EnigmaSnapshot:
    step: int
    input_letter: str
    output_letter: str
    rotor_positions: str
    path: tuple[str, ...]

    def summary(self) -> str:
        return f"step {self.step}: {self.input_letter} -> {self.output_letter} | rotors {self.rotor_positions}"


**Algorithm engine.** Define `EnigmaMachine`, the class that runs the main simulation or algorithm.


In [ ]:
class EnigmaMachine:
    def __init__(self, rotors: list[Rotor], reflector: Reflector, plugboard: Plugboard | None = None):
        if len(rotors) != 3:
            raise ValueError("This lesson simulator expects exactly three rotors.")
        self.rotors = rotors
        self.reflector = reflector
        self.plugboard = plugboard or Plugboard()
        self.snapshots: list[EnigmaSnapshot] = []

    def rotor_positions(self) -> str:
        return "".join(rotor.position_letter for rotor in self.rotors)

    def step_rotors(self) -> None:
        left, middle, right = self.rotors
        if middle.at_notch():
            left.step()
            middle.step()
        elif right.at_notch():
            middle.step()
        right.step()

    def encode_letter(self, letter: str) -> str:
        clean_letter = letter.upper()
        if clean_letter not in ALPHABET:
            return letter

        self.step_rotors()
        path = [f"input {clean_letter}", f"rotors step to {self.rotor_positions()}"]

        signal = self.plugboard.swap(clean_letter)
        path.append(f"plugboard in: {clean_letter} -> {signal}")

        for rotor in reversed(self.rotors):
            before = signal
            signal = rotor.encode_forward(signal)
            path.append(f"rotor {rotor.spec.name} forward: {before} -> {signal}")

        before_reflector = signal
        signal = self.reflector.reflect(signal)
        path.append(f"reflector: {before_reflector} -> {signal}")

        for rotor in self.rotors:
            before = signal
            signal = rotor.encode_backward(signal)
            path.append(f"rotor {rotor.spec.name} backward: {before} -> {signal}")

        output = self.plugboard.swap(signal)
        path.append(f"plugboard out: {signal} -> {output}")

        snapshot = EnigmaSnapshot(
            step=len(self.snapshots) + 1,
            input_letter=clean_letter,
            output_letter=output,
            rotor_positions=self.rotor_positions(),
            path=tuple(path),
        )
        self.snapshots.append(snapshot)
        return output

    def encode_message(self, message: str) -> str:
        output: list[str] = []
        for character in message.upper():
            if character in ALPHABET:
                output.append(self.encode_letter(character))
            else:
                output.append(character)
        return "".join(output)


**Builder helper.** Define `make_enigma_machine`, which prepares reusable examples or traces.


In [ ]:
ROTOR_SPECS = {
    "I": RotorSpec("I", "EKMFLGDQVZNTOWYHXUSPAIBRCJ", "Q"),
    "II": RotorSpec("II", "AJDKSIRUXBLHWTMCQGZNPYFVOE", "E"),
    "III": RotorSpec("III", "BDFHJLCPRTXVZNYEIWGAKMUSQO", "V"),
}

REFLECTOR_B = Reflector("YRUHQSLDPXNGOKMIEBFZCWVJAT")

def make_enigma_machine(positions: str, plug_pairs: tuple[str, ...] = ()) -> EnigmaMachine:
    clean_positions = normalize_message(positions)
    if len(clean_positions) != 3:
        raise ValueError("Use three rotor position letters, like 'KQF'.")
    rotors = [
        Rotor.from_letter(ROTOR_SPECS["I"], clean_positions[0]),
        Rotor.from_letter(ROTOR_SPECS["II"], clean_positions[1]),
        Rotor.from_letter(ROTOR_SPECS["III"], clean_positions[2]),
    ]
    return EnigmaMachine(rotors=rotors, reflector=REFLECTOR_B, plugboard=Plugboard(plug_pairs))


## 3. Part 1: Encrypt and Decrypt a Message

The sender and receiver must use the same rotor starting positions and plugboard pairs.

Here we use `KQF` as the daily setting. The message is not secure by modern standards, but the mechanics are real enough to see why Enigma was powerful.

In [2]:
daily_positions = "KQF"
plug_pairs = ("AV", "BS", "CG", "DL", "FU", "HZ")
message = "WEATHER REPORT AT DAWN"

sender = make_enigma_machine(daily_positions, plug_pairs)
ciphertext = sender.encode_message(message)

receiver = make_enigma_machine(daily_positions, plug_pairs)
recovered_message = receiver.encode_message(ciphertext)

print(f"Rotor positions: {daily_positions}")
print(f"Plugboard pairs: {', '.join(plug_pairs)}")
print(f"Plaintext : {message}")
print(f"Ciphertext: {ciphertext}")
print(f"Decoded   : {recovered_message}")

Rotor positions: KQF
Plugboard pairs: AV, BS, CG, DL, FU, HZ
Plaintext : WEATHER REPORT AT DAWN
Ciphertext: LSLRDZQ PCEPKS RN KBBW
Decoded   : WEATHER REPORT AT DAWN


## 4. Trace One Letter

A single letter takes a full round trip:

`plugboard -> right rotor -> middle rotor -> left rotor -> reflector -> left rotor -> middle rotor -> right rotor -> plugboard`

<details>
<summary>Rotor stepping detail</summary>

The rotors step before the letter is encoded. That means the first visible rotor position is already one move after the starting key.

</details>

In [3]:
trace_machine = make_enigma_machine(daily_positions, plug_pairs)
first_output = trace_machine.encode_letter("W")
trace_snapshot = trace_machine.snapshots[-1]

print(trace_snapshot.summary())
print()
for event in trace_snapshot.path:
    print(event)

step 1: W -> L | rotors KQG

input W
rotors step to KQG
plugboard in: W -> W
rotor III forward: W -> Z
rotor II forward: Z -> M
rotor I forward: M -> R
reflector: R -> B
rotor I backward: B -> U
rotor II backward: U -> N
rotor III backward: N -> D
plugboard out: D -> L


## 5. Part 2: A Simplified Bombe

The real Bombe used electrical logic to search huge numbers of Enigma settings. Our notebook version keeps the idea but simplifies the scale:

1. We intercept a ciphertext.
2. We guess that a word like `WETTER` appears somewhere in the plaintext.
3. We reject crib positions that are impossible.
4. We try rotor starting positions until the crib appears in the decrypted text.

<details>
<summary>Historical simplification</summary>

This is not a full Bombe replica. The real machines handled rotor orders, plugboard constraints, menus, and many simultaneous logical contradictions. This version focuses on the core search idea.

</details>

**Object model.** Define `BombeCandidate`, the named objects used by the next examples.


In [ ]:
@dataclass(frozen=True)
class BombeCandidate:
    positions: str
    plaintext: str
    crib_index: int


**Object model.** Define `SimpleBombe`, the named objects used by the next examples.


In [ ]:
class SimpleBombe:
    def __init__(self, plug_pairs: tuple[str, ...] = ()): 
        self.plug_pairs = plug_pairs

    def possible_crib_offsets(self, ciphertext: str, crib: str) -> list[int]:
        clean_ciphertext = normalize_message(ciphertext)
        clean_crib = normalize_message(crib)
        offsets: list[int] = []

        for start in range(len(clean_ciphertext) - len(clean_crib) + 1):
            window = clean_ciphertext[start : start + len(clean_crib)]
            has_self_encryption_conflict = any(
                cipher_letter == crib_letter
                for cipher_letter, crib_letter in zip(window, clean_crib)
            )
            if not has_self_encryption_conflict:
                offsets.append(start)

        return offsets

    def search(self, ciphertext: str, crib: str, limit: int = 5) -> tuple[int, list[BombeCandidate]]:
        clean_ciphertext = normalize_message(ciphertext)
        clean_crib = normalize_message(crib)
        offsets = self.possible_crib_offsets(clean_ciphertext, clean_crib)
        candidates: list[BombeCandidate] = []
        tested_settings = 0

        for letters in product(ALPHABET, repeat=3):
            positions = "".join(letters)
            trial_machine = make_enigma_machine(positions, self.plug_pairs)
            plaintext = normalize_message(trial_machine.encode_message(clean_ciphertext))
            tested_settings += 1

            for offset in offsets:
                if plaintext[offset : offset + len(clean_crib)] == clean_crib:
                    candidates.append(BombeCandidate(positions, plaintext, offset))
                    break

            if len(candidates) >= limit:
                break

        return tested_settings, candidates


## 6. Intercept a Message

For the Bombe demo, we will create a secret message using unknown rotor positions. The attacker knows the machine model and guesses the crib `WETTER`, but does not know the starting positions.

To keep the search fast and focused, this demo searches rotor positions only and leaves the plugboard empty.

In [7]:
secret_positions = "BCK"
secret_plaintext = "WETTERREPORTDAWN"

secret_machine = make_enigma_machine(secret_positions)
intercepted_ciphertext = secret_machine.encode_message(secret_plaintext)

print(f"Intercepted ciphertext: {intercepted_ciphertext}")
print("Known crib guess     : WETTER")
print("Rotor positions are hidden from the attacker.")

Intercepted ciphertext: SMZINWLVYTNZZEPM
Known crib guess     : WETTER
Rotor positions are hidden from the attacker.


## 7. Run the Bombe Search

The key Enigma clue: a letter never encrypts to itself. If a crib alignment would force plaintext `E` to match ciphertext `E` in the same position, that alignment can be eliminated immediately.

After that quick filter, the simplified Bombe tries rotor positions and checks whether the crib appears.

In [8]:
bombe = SimpleBombe()
possible_offsets = bombe.possible_crib_offsets(intercepted_ciphertext, "WETTER")
tested_settings, candidates = bombe.search(intercepted_ciphertext, "WETTER", limit=1)

print(f"Ciphertext length: {len(intercepted_ciphertext)}")
print(f"Crib offsets after self-encryption filter: {possible_offsets}")
print(f"Rotor settings tested before first hit: {tested_settings}")
print()

for candidate in candidates:
    print(
        f"positions={candidate.positions} "
        f"crib_index={candidate.crib_index} "
        f"plaintext={candidate.plaintext}"
    )

Ciphertext length: 16
Crib offsets after self-encryption filter: [0, 1, 2, 3, 4, 8, 10]
Rotor settings tested before first hit: 739

positions=BCK crib_index=0 plaintext=WETTERREPORTDAWN


## 8. Playground: Rotor Drift and Wrong Keys

The same letter does not always encrypt the same way because the rotors move.

A wrong starting position also decrypts into nonsense, even when the machine wiring is correct.

In [9]:
drift_machine = make_enigma_machine("AAA")
drift_ciphertext = drift_machine.encode_message("AAAAAAAAAA")

print(f"Encrypting AAAAAAAAAA from AAA gives: {drift_ciphertext}")
print("Rotor positions after each keypress:")
for snapshot in drift_machine.snapshots:
    print(f"step {snapshot.step:>2}: output={snapshot.output_letter} rotors={snapshot.rotor_positions}")

wrong_key_machine = make_enigma_machine("BCL")
wrong_plaintext = wrong_key_machine.encode_message(intercepted_ciphertext)

print("\nDecrypt intercepted ciphertext with wrong setting BCL:")
print(wrong_plaintext)

Encrypting AAAAAAAAAA from AAA gives: BDZGOWCXLT
Rotor positions after each keypress:
step  1: output=B rotors=AAB
step  2: output=D rotors=AAC
step  3: output=Z rotors=AAD
step  4: output=G rotors=AAE
step  5: output=O rotors=AAF
step  6: output=W rotors=AAG
step  7: output=C rotors=AAH
step  8: output=X rotors=AAI
step  9: output=L rotors=AAJ
step 10: output=T rotors=AAK

Decrypt intercepted ciphertext with wrong setting BCL:
QWWYEUXHKOXDGIKR


## 9. What You Should Remember

- Enigma is a rotor cipher: the substitution changes every keypress.
- The same settings both encrypt and decrypt because the reflector makes the path reversible.
- The plugboard adds another substitution layer before and after the rotors.
- A crib is a guessed plaintext fragment that helps eliminate impossible keys.
- The Bombe idea is search plus contradiction: throw away settings that cannot explain the crib, then test the survivors.

<details>
<summary>Security note</summary>

This is a historical learning simulator, not a modern security tool. Real cybersecurity uses public, peer-reviewed algorithms with strong keys, careful protocols, and modern threat models.

</details>

## Visual Trace + Rigor Studio

**Problem frame.** Understand encryption as a deterministic state machine.

**Interactive animation target.** Animate rotor positions, signal path, reflection, and stepping after each character.

**Correctness handle.** For a fixed state, Enigma maps each letter through a reversible wiring path.

**Complexity handle.** Encryption is O(message length), while brute-force key search grows with the key space.

**Failure mode to test.** Operational patterns and repeated settings can leak structure despite strong-looking machinery.

**Studio task.** Encrypt the same letter repeatedly and explain why outputs change.


In [ ]:
from pathlib import Path
import sys

for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "courseware").exists():
        sys.path.insert(0, str(candidate))
        break

from courseware import AlgorithmPlayer, AlgorithmTrace, TraceStep, render_trace_table

# Convert the implementation above into snapshots:
# trace = AlgorithmTrace("Topic trace")
# trace.append("start", {"your_state": ...}, "What changed?", invariant="What remains true?")
# AlgorithmPlayer(trace, your_renderer).display()
print("Use AlgorithmTrace to turn this notebook's algorithm into a step-by-step visual player.")
